# BinSense — M5: Siamese Similarity Model (the graded core)

**Task (per Objectives + the evaluator template `docs/Copy of Week 4 & 5.ipynb`):**
verify whether two bin images contain the **same item**. We train a **Siamese network**
on **positive pairs (same ASIN)** vs **negative pairs (different ASIN)**, compare
**VGG16 vs ResNet50** backbones, and verify with **ROC + threshold optimization +
confusion matrix**. The learned distance threshold is what the UI uses to answer
"is the ordered item present in this bin?".

*This replaces the detection-first path (see `docs/M3b_FINDINGS.md`): the template's
core needs no bounding boxes — it works on whole bin images, and identity comes from
the single-ASIN bins.* Framework is **TensorFlow/Keras** to match the template.
Pair logic lives in `tools/similarity/pairs.py` (tested); the model + eval are here.

In [ ]:
# Cell 1: Bootstrap — path resolution + data/code split
import sys, os, subprocess
from pathlib import Path
GITHUB_URL = 'https://github.com/rishib09/AmazonBinSense.git'
BRANCH     = 'm5-siamese'
DRIVE_ROOT = '/content/drive/MyDrive/Interview Kickstart/Capstone Project/Amazon BinSense'
LOCAL_DATA = r'G:\My Drive\Interview Kickstart\Capstone Project\Amazon BinSense\data'
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/AmazonBinSense')
    if (PROJECT_ROOT / '.git').exists():
        subprocess.run(['git','-C',str(PROJECT_ROOT),'fetch','origin',BRANCH], check=False)
        subprocess.run(['git','-C',str(PROJECT_ROOT),'checkout',BRANCH], check=False)
        subprocess.run(['git','-C',str(PROJECT_ROOT),'pull','origin',BRANCH,'--ff-only'], check=False)
    else:
        subprocess.run(['git','clone','--branch',BRANCH,GITHUB_URL,str(PROJECT_ROOT)], check=True)
    os.environ['BINSENSE_DATA_DIR'] = str(Path(DRIVE_ROOT) / 'data')
except ImportError:
    IN_COLAB = False
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    if not os.getenv('BINSENSE_DATA_DIR') and Path(LOCAL_DATA).exists():
        os.environ['BINSENSE_DATA_DIR'] = LOCAL_DATA
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Running in:', 'Google Colab' if IN_COLAB else 'Local', '| ROOT:', PROJECT_ROOT)

In [ ]:
# Cell 2: Build the Siamese pair index (engine = tools/similarity/pairs.py)
import numpy as np, pandas as pd, json
import tensorflow as tf
from utils.env_utils import setup_env
from tools.similarity.pairs import build_pairs, summarize

cfg = setup_env(verbose=True)
IMAGES_DIR = cfg.images_dir
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

NEG_PER_POS = 1          # 50/50 balance
train_pairs, val_pairs, bins_df = build_pairs(cfg.splits_dir, split='seed',
                                              val_frac=0.2, neg_per_pos=NEG_PER_POS, seed=42)
print(json.dumps(summarize(train_pairs, val_pairs, bins_df), indent=2))

## Step 1 — The pairs
Built from single-ASIN bins with an **ASIN-disjoint** train/val split (val ASINs are unseen in training). Positives are same-ASIN: `cross` = two different bins of one product (real intra-class variation), `aug` = two augmented views of one bin (most positives, given 289/317 ASINs appear in only one bin).

In [ ]:
# Cell 3: Look at a few pairs (identity signal is the whole game)
import matplotlib.pyplot as plt
def show_pairs(dfp, n=3, title=''):
    dfp = dfp.head(50)
    pos = dfp[dfp.label==1].head(n); neg = dfp[dfp.label==0].head(n)
    rows = list(pos.itertuples()) + list(neg.itertuples())
    fig, ax = plt.subplots(len(rows), 2, figsize=(5, 2.4*len(rows)))
    for i, r in enumerate(rows):
        for j, bid in enumerate([r.bin_a, r.bin_b]):
            img = plt.imread(str(IMAGES_DIR / f'{bid}.jpg'))
            ax[i, j].imshow(img); ax[i, j].axis('off')
            ax[i, j].set_title(f'{bid}  ({"SAME" if r.label==1 else "DIFF"}/{r.kind})', fontsize=8)
    plt.suptitle(title); plt.tight_layout(); plt.show()
show_pairs(train_pairs, 3, 'Sample training pairs — positives (same item) then negatives')

## Step 2 — Image pipeline

In [ ]:
# Cell 4: tf.data image pipeline (224x224, light augmentation, backbone preprocess)
# NOTE: augmentation is applied to BOTH images in EVERY pair (train and val). For the
# many "aug" positives (bin_a == bin_b) this yields two DIFFERENT views, so a positive
# pair is never the trivially-identical image — which would otherwise inflate the ROC.
def make_dataset(pairs_df, backbone='vgg16', batch=16, shuffle=False):
    pre = (tf.keras.applications.vgg16.preprocess_input if backbone == 'vgg16'
           else tf.keras.applications.resnet50.preprocess_input)
    a = [str(IMAGES_DIR / f'{b}.jpg') for b in pairs_df['bin_a']]
    b = [str(IMAGES_DIR / f'{b}.jpg') for b in pairs_df['bin_b']]
    y = pairs_df['label'].astype('float32').values

    def load(path):
        img = tf.image.decode_jpeg(tf.io.read_file(path), channels=3)
        return tf.image.resize(img, (224, 224))

    def aug(img):
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, 0.10)
        img = tf.image.random_contrast(img, 0.9, 1.1)
        return tf.clip_by_value(img, 0.0, 255.0)

    def prep(pa, pb, label):
        ia, ib = aug(load(pa)), aug(load(pb))
        return (pre(ia), pre(ib)), label

    ds = tf.data.Dataset.from_tensor_slices((a, b, y))
    if shuffle:
        ds = ds.shuffle(len(y), seed=42)
    return ds.map(prep, num_parallel_calls=tf.data.AUTOTUNE).batch(batch).prefetch(tf.data.AUTOTUNE)

print('pipeline ready')

## Step 3 — Siamese architecture
A shared backbone maps each image to an L2-normalized embedding; the model outputs the **Euclidean distance** between the pair. **Contrastive loss** pulls same-item pairs together and pushes different-item pairs past a margin.

In [ ]:
# Cell 5: Siamese model — shared backbone embedding, Euclidean distance, contrastive loss
def build_embedding(backbone='vgg16', input_shape=(224, 224, 3), emb_dim=128, freeze=True):
    if backbone == 'vgg16':
        base = tf.keras.applications.VGG16(include_top=False, weights='imagenet', input_shape=input_shape)
    else:
        base = tf.keras.applications.ResNet50(include_top=False, weights='imagenet', input_shape=input_shape)
    base.trainable = not freeze          # transfer learning: freeze backbone first
    inp = tf.keras.Input(input_shape)
    x = base(inp, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dense(emb_dim)(x)
    x = tf.keras.layers.Lambda(lambda t: tf.math.l2_normalize(t, axis=1))(x)  # unit sphere
    return tf.keras.Model(inp, x, name=f'emb_{backbone}')

def build_siamese(embedding, input_shape=(224, 224, 3)):
    a, b = tf.keras.Input(input_shape), tf.keras.Input(input_shape)
    ea, eb = embedding(a), embedding(b)
    dist = tf.keras.layers.Lambda(
        lambda t: tf.sqrt(tf.reduce_sum(tf.square(t[0] - t[1]), axis=1, keepdims=True) + 1e-9)
    )([ea, eb])
    return tf.keras.Model([a, b], dist, name='siamese')

def contrastive_loss(margin=1.0):
    # label 1 = same item -> pull distance to 0 ; label 0 = different -> push past margin
    def loss(y_true, y_pred):
        y = tf.cast(tf.reshape(y_true, (-1,)), tf.float32)
        d = tf.reshape(y_pred, (-1,))
        return tf.reduce_mean(y * tf.square(d) + (1.0 - y) * tf.square(tf.maximum(margin - d, 0.0)))
    return loss

print('model builders ready')

## Step 4 — Train (VGG16, frozen backbone = transfer learning)

In [ ]:
# Cell 6: Train the VGG16 Siamese (frozen backbone, transfer learning)
EPOCHS, BATCH = 20, 16
tf.keras.utils.set_random_seed(42)

emb_vgg = build_embedding('vgg16', freeze=True)
siam_vgg = build_siamese(emb_vgg)
siam_vgg.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=contrastive_loss(margin=1.0))

train_ds = make_dataset(train_pairs, 'vgg16', BATCH, shuffle=True)
val_ds   = make_dataset(val_pairs,   'vgg16', BATCH, shuffle=False)

hist_vgg = siam_vgg.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)])

SAVE = cfg.models_dir / 'siamese_vgg16'; SAVE.mkdir(parents=True, exist_ok=True)
siam_vgg.save_weights(str(SAVE / 'siamese_vgg16.weights.h5'))
print('saved ->', SAVE)

plt.plot(hist_vgg.history['loss'], label='train'); plt.plot(hist_vgg.history['val_loss'], label='val')
plt.title('VGG16 Siamese — contrastive loss'); plt.xlabel('epoch'); plt.legend(); plt.show()

## Step 5 — Verify (ROC / threshold / confusion) — the evaluator's method
Distance is thresholded to decide same/different. ROC-AUC measures separability independent of threshold; Youden's J picks the operating point; the confusion matrix shows errors there.

In [ ]:
# Cell 7: Verify — ROC + threshold optimization + confusion matrix (the graded method)
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

def evaluate(model, pairs_df, backbone, name):
    ds = make_dataset(pairs_df, backbone, batch=16, shuffle=False)
    d = model.predict(ds, verbose=0).ravel()          # euclidean distances
    y = pairs_df['label'].values.astype(int)
    auc = roc_auc_score(y, -d)                          # smaller distance => more similar
    fpr, tpr, thr = roc_curve(y, -d)
    best_dist = -thr[np.argmax(tpr - fpr)]              # Youden's J optimal threshold
    pred = (d < best_dist).astype(int)                  # predict SAME if distance < thr
    cm = confusion_matrix(y, pred); acc = float((pred == y).mean())

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(fpr, tpr, label=f'AUC={auc:.3f}'); ax[0].plot([0,1],[0,1],'k--',lw=1)
    ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR'); ax[0].legend()
    ax[0].set_title(f'{name} ROC')
    ConfusionMatrixDisplay(cm, display_labels=['diff','same']).plot(ax=ax[1], colorbar=False)
    ax[1].set_title(f'{name} @ dist<{best_dist:.3f}  (acc={acc:.2f})')
    plt.tight_layout(); plt.show()
    return {'backbone': backbone, 'auc': round(auc,3), 'threshold': round(float(best_dist),3), 'acc': round(acc,3)}

res_vgg = evaluate(siam_vgg, val_pairs, 'vgg16', 'VGG16')
print(res_vgg)

## Step 6 — Backbone comparison: ResNet50

In [ ]:
# Cell 8: Backbone comparison — ResNet50 (same pipeline, same eval)
emb_res = build_embedding('resnet50', freeze=True)
siam_res = build_siamese(emb_res)
siam_res.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=contrastive_loss(margin=1.0))

train_ds_r = make_dataset(train_pairs, 'resnet50', 16, shuffle=True)
val_ds_r   = make_dataset(val_pairs,   'resnet50', 16, shuffle=False)
hist_res = siam_res.fit(train_ds_r, validation_data=val_ds_r, epochs=20,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)])
SAVE_R = cfg.models_dir / 'siamese_resnet50'; SAVE_R.mkdir(parents=True, exist_ok=True)
siam_res.save_weights(str(SAVE_R / 'siamese_resnet50.weights.h5'))

res_res = evaluate(siam_res, val_pairs, 'resnet50', 'ResNet50')
print(res_res)

## Step 7 — Decision & handoff
Pick the higher-AUC backbone and its threshold; save to `models/siamese_decision.json` for the UI/inference step.

In [ ]:
# Cell 9: Pick backbone + threshold -> handoff to productionize / UI
results = pd.DataFrame([res_vgg, res_res]).sort_values('auc', ascending=False)
display(results)
best = results.iloc[0]
decision = {'best_backbone': best['backbone'], 'val_auc': float(best['auc']),
            'similarity_threshold_distance': float(best['threshold']),
            'note': 'UI verifies "same item" when embedding distance < threshold'}
out = cfg.models_dir / 'siamese_decision.json'
out.write_text(json.dumps(decision, indent=2))
print('WROTE', out); print(json.dumps(decision, indent=2))

## Handoff to productionize

- **Model:** best of VGG16 / ResNet50 (by val ROC-AUC), weights on Drive under `models/`.
- **Threshold:** the saved distance cutoff = the UI's "same item" decision boundary.
- **Inference flow for the UI:** user picks an ordered ASIN → load a reference image of
  that ASIN → embed both it and the queried bin image → if distance < threshold, the item
  is verified present. Quantity handling and the Streamlit/Gradio UI come next.
- **Known limits to state honestly:** small identity set (317 ASINs, most single-bin), so
  positives lean on augmentation; report val-AUC with that caveat. Frozen backbone first;
  unfreezing top blocks is the obvious next lever if AUC is low.